# TS-SatFire: 이틀 입력과 다음 날 NDVI·신규 소실 비교

## 목적과 실행 범위

- FP 사건에 대해 **모든 선택 파일의 원래 shape·CRS·transform**을 조사합니다.
- 입력은 **t−1, t 두 날**, 예측/비교 대상은 **t+1**입니다. 예측 모델을 학습하는 노트북은 아닙니다.
- NDVI(t)와 NDVI(t+1), 변화량을 시각화하며 **확산 없음·확산 있음·라벨 불명**을 구분합니다.
- 기본 설정은 HPC 실제 데이터용입니다. 작성 환경에서는 해당 HPC 파일에 접근할 수 없으므로
  실제 NDVI/영상 크기의 결론은 실행 후 확인해야 합니다. 검증용 합성 예제는 DEMO_MODE=True로 별도 실행할 수 있습니다.

**발화일 주의:** ROI 시작일, 첫 위성 관측일, 실제 발화일은 동일하다고 가정하지 않습니다.
첫 관측일부터 자료가 시작하면 발화일 당일의 이틀 입력은 구성할 수 없을 수 있습니다.
기본 anchor는 이틀 이력과 다음 날 관측이 있는 첫 날짜이며 실제 발화일이라고 부르지 않습니다.

출처: [TS-SatFire 논문](https://arxiv.org/html/2412.11555v1),
저장소 `legacy/roi`·`legacy/scripts/dataset_gen_pred.py`, 지정한 HPC GeoTIFF.

## 환경

HPC에서 다음 환경을 준비하고 Jupyter의 커널을 **Python (satfire-audit)**로 선택합니다.
Jupyter는 해당 HPC의 지원 방식(Open OnDemand 등)으로 시작하십시오. GPU는 필요 없습니다.

```bash
conda activate satfire-audit
conda install -c conda-forge jupyterlab ipykernel pandas matplotlib nbformat nbclient -y
python -m ipykernel install --user --name satfire-audit --display-name "Python (satfire-audit)"
```

원본 파일은 수정하지 않습니다. 보고서는 저장소의 `audit_results/ndvi_two_day` 아래에 생성됩니다.
같은 설정 재실행 시 이 폴더의 진단 CSV/그림만 갱신합니다. 원본 밖의 경로인지 검사합니다.

In [ ]:
from pathlib import Path
import sys, os, json, datetime as dt
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'scripts'/'satfire_explore.py').is_file()), None)
if REPO is None:
    raise RuntimeError('이 저장소 루트 또는 notebooks 폴더에서 실행하세요.')
sys.path.insert(0, str(REPO/'scripts'))
from fp_audit import read_fp_split
from satfire_explore import (create_demo_dataset, discover_fp, header_inventory,
    alignment_report, date_index, unique_path, header, grid_reasons, band_stats,
    read_ndvi, progression, progression_series)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

## 1. 설정

제공 NDVI는 전체 27채널 표에서 9번, 별도 FirePred에서는 **1번 밴드라는 전제**를
사용합니다. 실제 밴드 태그와 원본 로더를 아래 표와 대조하십시오.
NDVI_SCALE=None은 파일의 scale/offset을 사용합니다. 값이 NDVI 범위를 크게 벗어나면
그림을 만들지 않고 스케일 확인을 요청합니다. **값을 보고 근거 없이 0~1 정규화하지 마십시오.**

BA_RULE은 기본 None입니다. 실제 소실 라벨을 확인하기 전에는 '확산 없음'을 단정하지 않습니다.
유한 날짜코드=소실/NaN=비소실 인코딩이 확인되면 'finite', 0/양수 인코딩이 확인되면
'positive'로 바꾸고 다시 실행하세요. 밴드 전체 NaN은 보수적으로 라벨 불명으로 처리합니다.

직접 NDVI를 계산하려면 NDVI_SOURCE='computed'와 REFLECTANCE_CONFIRMED=True가 필요합니다.
VIIRS 입력은 Level-1B 기원이며 복사휘도/스케일을 확인하지 않고 I2/I1 비율을 적용하면 안 됩니다.

In [ ]:
DEMO_MODE = False                         # 합성 데이터: 결과를 실제 산불로 해석하지 않음
DATA_ROOT = Path('/home/jlc3q/data/SatFire/ts-satfire')
ROI_DIR = REPO/'legacy'/'roi'
SPLIT_SCRIPT = REPO/'legacy'/'scripts'/'dataset_gen_pred.py'
EVENT_ID = None                            # 예: 실제 CSV ID 문자열, None=첫 구성 가능 사건
ANCHOR_DATE = None                         # 예: '2021-08-03', 이 날짜가 입력 마지막 날 t
SCAN_ALL_HEADERS = True                    # False이면 선택 사건만 검사, 전체 확인이라고 부르지 않음
NDVI_SOURCE = 'provided'                    # 'provided' 또는 'computed'
PROVIDED_NDVI_BAND = 1                     # 1-based; FirePred 파일 기준
NDVI_SCALE = None                          # 확인 후 필요하면 명시. 자동 추정하지 않음
NDVI_OFFSET = None
REFLECTANCE_CONFIRMED = False
BA_BAND = 8                               # 1-based 후보; 실제 확인 필요
BA_RULE = None                            # None / 'finite' / 'positive'
SHOW_STATUS_EXAMPLES = True

In [ ]:
if DEMO_MODE:
    import tempfile
    DEMO_DIR = Path(tempfile.mkdtemp(prefix='satfire_ndvi_demo_'))
    DATA_ROOT, ROI_DIR, SPLIT_SCRIPT = create_demo_dataset(DEMO_DIR)
    BA_RULE, NDVI_SCALE = 'finite', 1.0
    print('SYNTHETIC DEMO — NOT ACTUAL WILDFIRE DATA:', DATA_ROOT)
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'HPC 데이터에 접근할 수 없습니다: {DATA_ROOT}')
REPORT_DIR = REPO/'audit_results'/('ndvi_two_day_demo' if DEMO_MODE else 'ndvi_two_day')
if REPORT_DIR.resolve() == DATA_ROOT.resolve() or DATA_ROOT.resolve() in REPORT_DIR.resolve().parents:
    raise ValueError('보고서는 원본 데이터 밖에 저장해야 합니다.')
REPORT_DIR.mkdir(parents=True, exist_ok=True)
records, split_provenance = read_fp_split(ROI_DIR, SPLIT_SCRIPT)
files = discover_fp(DATA_ROOT, records)
if not files:
    raise RuntimeError('FP 분할 ID와 일치하는 GeoTIFF가 없습니다. 경로와 사건 폴더명을 확인하세요.')
file_table = pd.DataFrame(files)
display(file_table.groupby(['split', 'modality']).size().rename('files').to_frame())
print('Declared FP split:', split_provenance['declared_counts'])

## 2. 사건·이틀 입력 선택

선택은 확산 유무와 관계없이 파일 가용성만으로 합니다. 입력 마지막 날짜 t와 t+1의
NDVI를 비교합니다. t+1의 NDVI는 **사후 진단에만 사용하며 예측 입력에 넣지 않습니다.**

In [ ]:
def available_anchors(index):
    modality = 'FirePred' if NDVI_SOURCE == 'provided' else 'VIIRS_Day'
    return [d for d in sorted(index['VIIRS_Day'])
            if all(unique_path(index, 'VIIRS_Day', d+dt.timedelta(days=k)) for k in [-1, 0, 1])
            and all(unique_path(index, modality, d+dt.timedelta(days=k)) for k in [-1, 0, 1])]

if EVENT_ID is None:
    for candidate in sorted(file_table['event_id'].unique()):
        try:
            candidate_index = date_index(files, candidate)
        except ValueError:
            continue
        if available_anchors(candidate_index):
            EVENT_ID = candidate
            break
if EVENT_ID is None or EVENT_ID not in records:
    raise ValueError('이틀 이력과 다음 날 관측이 있는 사건을 지정하세요.')
index = date_index(files, EVENT_ID)
anchors = available_anchors(index)
if not anchors:
    raise ValueError('선택 사건에서 t−1, t, t+1 연속 관측을 구성할 수 없습니다.')
t = dt.date.fromisoformat(ANCHOR_DATE) if ANCHOR_DATE else anchors[0]
if t not in anchors:
    raise ValueError(f'지정 날짜는 이틀 입력·다음 날 비교가 불가능합니다. 예시: {anchors[:8]}')
dates = [t-dt.timedelta(days=1), t, t+dt.timedelta(days=1)]
display(pd.DataFrame([dict(**records[EVENT_ID], selected_t=str(t))]))
print('Input:', dates[:2], ' | Retrospective next-day comparison:', dates[2])

## 3. 영상 크기가 실제로 같은지 확인

원본 GeoTIFF 크기와 모델에서 자르는 256×256 크기는 구분합니다.
아래는 선택 범위의 **모든 파일 헤더**를 읽습니다. 첫 실행은 파일 수에 따라 시간이 걸릴 수 있습니다.
동일 shape라도 CRS가 없거나 transform이 다르면 같은 위치라는 확인은 끝나지 않은 것입니다.
자동 재투영·강제 CRS 지정·원본 수정은 하지 않습니다.

In [ ]:
scope_files = files if SCAN_ALL_HEADERS else [r for r in files if r['event_id'] == EVENT_ID]
headers = header_inventory(scope_files)
header_table = pd.DataFrame(headers)
readable = header_table[header_table['error'].isna()].copy()
readable['height'] = readable['shape'].map(lambda x: x[0])
readable['width'] = readable['shape'].map(lambda x: x[1])
readable['crs_display'] = readable['crs'].fillna('MISSING')
display(readable.groupby(['modality', 'height', 'width', 'crs_display']).size().rename('files').to_frame())
n_shapes = readable[['height', 'width']].drop_duplicates().shape[0]
print('Scope:', 'ALL SELECTED FP FILES' if SCAN_ALL_HEADERS else 'SELECTED EVENT ONLY')
print('Requested/readable/unreadable:', len(scope_files), len(readable), len(headers)-len(readable))
print('Distinct spatial shapes:', n_shapes)
if len(readable) == len(scope_files) and n_shapes == 1:
    print('검사 범위의 모든 파일은 동일한 height/width입니다. 위치 정합은 아래에서 별도 확인합니다.')
else:
    print('크기가 여러 종류이거나 읽지 못한 파일이 있습니다. 동일 크기라고 확정하지 않습니다.')
alignment = pd.DataFrame(alignment_report(headers))
display(alignment.groupby(['modality', 'reasons']).size().rename('same_day_pairs').to_frame())
display(alignment[alignment['reasons'] != 'aligned'].head(12))
header_table.to_csv(REPORT_DIR/'all_fp_headers.csv', index=False)
alignment.to_csv(REPORT_DIR/'same_day_grid_alignment.csv', index=False)

## 4. 밴드·스케일·라벨 원시값 확인

현재 고른 3일의 값을 확인한 뒤 위 설정을 수정할 수 있습니다.
밴드명 부재나 스케일=1은 값의 의미가 검증됐다는 뜻이 아닙니다.
현재 파일의 전체 NaN을 비화재 0으로 바꾸지 않습니다.

In [ ]:
raw_rows = []
for d in dates:
    day_path = unique_path(index, 'VIIRS_Day', d)
    ndvi_path = unique_path(index, 'FirePred', d)
    for kind, path, band in [('BA candidate', day_path, BA_BAND),
                             ('provided NDVI candidate', ndvi_path, PROVIDED_NDVI_BAND)]:
        if path:
            try:
                raw_rows.append(dict(date=str(d), kind=kind, **band_stats(path, band)))
            except Exception as exc:
                raw_rows.append(dict(date=str(d), kind=kind, error=str(exc)))
raw_stats = pd.DataFrame(raw_rows)
display(raw_stats)
raw_stats.to_csv(REPORT_DIR/'selected_raw_band_stats.csv', index=False)

## 5. NDVI 비교와 변화량

색 범위는 날짜마다 고정합니다. 검증 가능한 동일 격자에서 공통 유효 픽셀만 차분합니다.
NDVI 감소는 구름·연무·시야각·소실 등에 의해서도 발생하므로 신규 소실 정답으로 사용하지 않습니다.
제공 NDVI 제품은 합성 관측일 수 있어 파일 날짜가 하루 달라도 값이 동일할 수 있습니다.

In [ ]:
def plot_pair(current, save_name=None):
    sample_dates = [current-dt.timedelta(days=1), current, current+dt.timedelta(days=1)]
    modality = 'FirePred' if NDVI_SOURCE == 'provided' else 'VIIRS_Day'
    paths = [unique_path(index, modality, d) for d in sample_dates]
    observations = [read_ndvi(p, source=NDVI_SOURCE, band=PROVIDED_NDVI_BAND,
        scale=NDVI_SCALE, offset=NDVI_OFFSET, reflectance_confirmed=REFLECTANCE_CONFIRMED) for p in paths]
    display(pd.DataFrame([dict(date=str(d), **{k:v for k,v in obs.items() if k != 'array'})
                          for d, obs in zip(sample_dates, observations)]))
    fig, axes = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)
    images = []
    for ax, date, obs in zip(axes[:3], sample_dates, observations):
        arr = obs['array']
        if arr is None:
            ax.text(.5, .5, obs['status'], ha='center', va='center', wrap=True, transform=ax.transAxes)
        else:
            images.append(ax.imshow(arr, vmin=-1, vmax=1, cmap='viridis'))
        ax.set_title(f'{date} | '+('input t-1' if date < current else 'input t' if date == current else 'observed t+1'))
        ax.set_xlabel('Column (native pixels)')
        ax.set_ylabel('Row (native pixels)')
    if images:
        fig.colorbar(images[0], ax=list(axes[:3]), shrink=.8, label='NDVI')
    before_path = unique_path(index, 'VIIRS_Day', current)
    after_path = unique_path(index, 'VIIRS_Day', sample_dates[2])
    change = progression(before_path, after_path, BA_BAND, BA_RULE)
    delta, common = None, None
    if observations[1]['array'] is not None and observations[2]['array'] is not None:
        reasons = grid_reasons(header(paths[1]), header(paths[2]))
        if not reasons:
            a, b = observations[1]['array'], observations[2]['array']
            common = np.isfinite(a) & np.isfinite(b)
            delta = np.where(common, b-a, np.nan)
            image = axes[3].imshow(delta, vmin=-2, vmax=2, cmap='RdBu_r')
            fig.colorbar(image, ax=axes[3], shrink=.8, label='NDVI(t+1) - NDVI(t)')
            overlay_ok = (change['new_mask'] is not None and before_path is not None
                and not grid_reasons(header(before_path), header(paths[1])))
            if overlay_ok and change['new_mask'].any() and not change['new_mask'].all():
                axes[3].contour(change['new_mask'].astype(float), levels=[.5], colors=['black'], linewidths=.8)
            print('Common valid NDVI pixels:', int(common.sum()), '/', common.size)
        else:
            axes[3].text(.5, .5, 'No pixel subtraction: '+', '.join(reasons), ha='center', va='center', wrap=True, transform=axes[3].transAxes)
    else:
        axes[3].text(.5, .5, 'NDVI unavailable: inspect raw values and scale', ha='center', va='center', wrap=True, transform=axes[3].transAxes)
    axes[3].set_title('Change; black contour = new BA if verified')
    axes[3].set_xlabel('Column (native pixels)')
    axes[3].set_ylabel('Row (native pixels)')
    fig.suptitle(f'{EVENT_ID} | '+('SYNTHETIC | ' if DEMO_MODE else '')+change['status'])
    if save_name:
        fig.savefig(REPORT_DIR/f'{save_name}.png', dpi=150)
    plt.show()
    print({k:v for k,v in change.items() if not isinstance(v, np.ndarray)})
    return observations, change

observations, selected_change = plot_pair(t, 'selected_ndvi_pair')

## 6. 확산 없는 날짜와 라벨 없는 날짜 구분

이 표는 선택 사건의 관측일마다 t→t+1을 점검합니다. 두 날 입력 이력이 없는 날짜도
표에 남기며 `two_day_history=False`로 구분합니다. 소실 라벨 비교 결과와 모델 입력 가용성은 별개입니다.

| 상태 | 의미 | 나중의 학습 처리 |
| --- | --- | --- |
| observed_no_increment | 비교 영역 전체에서 유효 라벨상 신규 소실 0 | 정상 음성 사례로 유지. 위험이 영구히 0이라는 뜻은 아님 |
| observed_increment | 유효 라벨상 신규 소실 >0 | 신규 소실=1, 비교 가능한 나머지=0 |
| partial_coverage_* | 일부 픽셀에서만 라벨 비교 가능 | 유효 픽셀에만 loss 적용. 나머지는 unknown |
| unknown_* | 인코딩·관측·라벨·격자 확인 불가 | 0으로 바꾸지 말고 제외/마스킹 |

전체 NaN이 실제 비소실인지 미주석인지 구별할 근거가 없으면 보수적으로 unknown입니다.
위 observed 상태도 지정한 인코딩하의 관측 결과이며 현장 진실을 보장하지 않습니다.

In [ ]:
series = pd.DataFrame(progression_series(index, BA_BAND, BA_RULE))
display(series)
display(series.groupby(['two_day_history', 'status']).size().rename('date_pairs').to_frame())
series.to_csv(REPORT_DIR/'progression_by_day.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 3), constrained_layout=True)
new_values = pd.to_numeric(series.get('new_pixels', pd.Series(np.nan, index=series.index)), errors='coerce')
ax.plot(pd.to_datetime(series['current_date']), new_values, marker='o', label='Newly burned pixels (NaN = unknown)')
ax.set(xlabel='Last input date t', ylabel='Newly burned pixels', title='Observed next-day increments; unknowns are gaps')
ax.grid(alpha=.2)
ax.legend()
plt.show()

if SHOW_STATUS_EXAMPLES:
    for status in ['observed_increment', 'observed_no_increment', 'unknown_missing_annotation']:
        examples = series[(series['status'] == status) & series['two_day_history']]
        if len(examples):
            d = dt.date.fromisoformat(examples.iloc[0]['current_date'])
            if d != t:
                print('Example:', status, d)
                plot_pair(d, 'example_'+status)

## 판단과 다음 단계

- shape가 같다는 결과와 CRS/transform까지 정합된다는 결과를 별도로 판단합니다.
- 확산 없는 유효 날짜도 유지합니다. 확산 날짜만 골라 학습하면 확률을 과대평가할 수 있습니다.
- 라벨 불명과 비소실을 구분한 다음, 학습·검증·시험 사건을 유지하여 모델을 구성합니다.
- NDVI(t+1)는 사후 비교·평가 전용입니다. 예측 입력에는 t까지의 정보만 사용합니다.
- NDVI 스케일과 BA 인코딩을 확인한 뒤 재실행해야 실제 값·확산 빈도를 해석할 수 있습니다.

이 노트북은 제공 BA의 일별 차이를 관찰합니다. legacy의 누적 AF/합집합 또는
SpaSE-UNet3D의 전처리·crop을 그대로 재현한 벤치마크는 아닙니다.

In [ ]:
provenance = dict(demo=DEMO_MODE, data_root=str(DATA_ROOT), event_id=EVENT_ID,
    anchor_date=str(t), input_days=2, ndvi_source=NDVI_SOURCE,
    provided_ndvi_band=PROVIDED_NDVI_BAND, ndvi_scale=NDVI_SCALE,
    ndvi_offset=NDVI_OFFSET, reflectance_confirmed=REFLECTANCE_CONFIRMED,
    ba_band=BA_BAND, ba_rule=BA_RULE, all_fp_headers_requested=SCAN_ALL_HEADERS,
    header_requested_count=len(scope_files), header_readable_count=len(readable),
    split_provenance=split_provenance)
(REPORT_DIR/'notebook_provenance.json').write_text(json.dumps(provenance, indent=2, ensure_ascii=False), encoding='utf-8')
print('Reports:', REPORT_DIR)